In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device: CPU")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
Device: CPU


In [ ]:
!pip install -q transformers datasets
!pip install -q torch-geometric
!pip install -q librosa soundfile
!pip install -q scikit-learn pandas matplotlib seaborn networkx pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.4 MB/s eta 0:00:00


In [ ]:
CONFIG = {
    "dataset": "FMA-small",
    "max_tracks": 800,
    "sample_rate": 22050,
    "segment_seconds": 5,
    "max_text_length": 128,
    "bert_model": "distilbert-base-uncased",
    "batch_size": 4,
    "epochs": 3,
    "learning_rate": 2e-5,
    "num_classes": 8
}

CONFIG

{'dataset': 'FMA-small',
 'max_tracks': 800,
 'sample_rate': 22050,
 'segment_seconds': 5,
 'max_text_length': 128,
 'bert_model': 'distilbert-base-uncased',
 'batch_size': 4,
 'epochs': 3,
 'learning_rate': 2e-05,
 'num_classes': 8}

# FMA

In [ ]:
!mkdir -p /content/fma

In [ ]:
!wget -O /content/fma/fma_metadata.zip \
"https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"

--2026-09-11 10:15:53--  https://os.unil.cloud.switch.ch/fma/fma_metadata.zip
Resolving os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)... 86.119.28.16, 2001:620:5ca1:201::214
Connecting to os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)|86.119.28.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358412441 (342M) [application/zip]
Saving to: ‘/content/fma/fma_metadata.zip’

/content/fma/fma_me 100%[===================>] 341.81M  25.5MB/s    in 14s     

2026-09-11 10:16:08 (23.7 MB/s) - ‘/content/fma/fma_metadata.zip’ saved [358412441/358412441]



In [ ]:
!wget -O /content/fma/fma_small.zip \
"https://os.unil.cloud.switch.ch/fma/fma_small.zip"

--2026-09-11 10:16:15--  https://os.unil.cloud.switch.ch/fma/fma_small.zip
Resolving os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)... 86.119.28.16, 2001:620:5ca1:201::214
Connecting to os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)|86.119.28.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7679594875 (7.2G) [application/zip]
Saving to: ‘/content/fma/fma_small.zip’

/content/fma/fma_sm 100%[===================>]   7.15G  26.7MB/s    in 4m 53s  

2026-09-11 10:21:09 (25.0 MB/s) - ‘/content/fma/fma_small.zip’ saved [7679594875/7679594875]



In [ ]:
!mkdir -p /content/fma/metadata
!unzip -q /content/fma/fma_metadata.zip -d /content/fma/metadata

In [ ]:
import pandas as pd

tracks = pd.read_csv(
    "/content/fma/metadata/fma_metadata/tracks.csv",
    header=[0, 1],
    index_col=0
)

print(tracks.shape)
tracks.head()

(106574, 52)


album                                                     \
         comments         date_created        date_released engineer   
track_id                                                               
2               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
3               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
5               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
10              0  2008-11-26 01:45:08  2008-02-06 00:00:00      NaN   
20              0  2008-11-26 01:45:05  2009-01-06 00:00:00      NaN   

                                                                          \
         favorites id                                information listens   
track_id                                                                   
2                4  1                                    <p></p>    6073   
3                4  1                                    <p></p>    6073   
5                4  1                                    <p></p>    6073   
10               4  6                                        NaN   47632   
20               2  4  <p> "spiritual songs" from Nicky Cook</p>    2710   

                        ...       track                         \
         producer tags  ... information interest language_code   
track_id                ...                                      
2             NaN   []  ...         NaN     4656            en   
3             NaN   []  ...         NaN     1470            en   
5             NaN   []  ...         NaN     1933            en   
10            NaN   []  ...         NaN    54881            en   
20            NaN   []  ...         NaN      978            en   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
2         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1293      NaN   
3         Attribution-NonCommercial-ShareAlike 3.0 Inter...     514      NaN   
5         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1151      NaN   
10        Attribution-NonCommercial-NoDerivatives (aka M...   50135      NaN   
20        Attribution-NonCommercial-NoDerivatives (aka M...     361      NaN   

                                                 
         number publisher tags            title  
track_id                                         
2             3       NaN   []             Food  
3             4       NaN   []     Electric Ave  
5             6       NaN   []       This World  
10            1       NaN   []          Freeway  
20            3       NaN   []  Spiritual Level  

[5 rows x 52 columns]

# Track/File Extraction

In [ ]:
import zipfile
import os

with zipfile.ZipFile("/content/fma/fma_small.zip", "r") as z:
    members = z.namelist()

available_ids = []

for member in members:
    if member.lower().endswith(".mp3"):
        filename = os.path.basename(member)
        try:
            track_id = int(os.path.splitext(filename)[0])
            available_ids.append(track_id)
        except ValueError:
            pass

available_ids = sorted(set(available_ids))

print("Tracks actually in FMA-small:", len(available_ids))

Tracks actually in FMA-small: 8000


In [ ]:
genre_column = ("track", "genre_top")

genres = [
    "Electronic",
    "Experimental",
    "Folk",
    "Hip-Hop",
    "Instrumental",
    "International",
    "Pop",
    "Rock"
]

available_tracks = tracks.loc[
    tracks.index.intersection(available_ids)
].copy()

selected_ids = []

for genre in genres:
    genre_ids = available_tracks[
        available_tracks[genre_column] == genre
    ].index[:100]

    selected_ids.extend(genre_ids.tolist())

print("Selected tracks:", len(selected_ids))

Selected tracks: 800


In [ ]:
import zipfile
import os
import shutil

audio_dir = "/content/fma/audio"
os.makedirs(audio_dir, exist_ok=True)

selected_ids_set = set(int(x) for x in selected_ids)

with zipfile.ZipFile("/content/fma/fma_small.zip", "r") as z:
    extracted = 0

    for member in z.namelist():

        if not member.lower().endswith(".mp3"):
            continue

        filename = os.path.basename(member)

        try:
            track_id = int(os.path.splitext(filename)[0])
        except ValueError:
            continue

        if track_id not in selected_ids_set:
            continue

        output_path = os.path.join(
            audio_dir,
            filename
        )

        if os.path.exists(output_path):
            continue

        with z.open(member) as source:
            with open(output_path, "wb") as target:
                shutil.copyfileobj(source, target)

        extracted += 1

        if extracted % 50 == 0:
            print(f"Extracted {extracted}/800")

print("Newly extracted:", extracted)

Extracted 50/800
Extracted 100/800
Extracted 150/800
Extracted 200/800
Extracted 250/800
Extracted 300/800
Extracted 350/800
Extracted 400/800
Extracted 450/800
Extracted 500/800
Extracted 550/800
Extracted 600/800
Extracted 650/800
Extracted 700/800
Extracted 750/800
Extracted 800/800
Newly extracted: 800


In [ ]:
selected_audio_count = sum(
    os.path.exists(
        os.path.join(
            audio_dir,
            f"{int(track_id):06d}.mp3"
        )
    )
    for track_id in selected_ids
)

print("Selected tracks:", len(selected_ids))
print("Selected audio files:", selected_audio_count)
print(
    "Missing:",
    len(selected_ids) - selected_audio_count
)

Selected tracks: 800
Selected audio files: 800
Missing: 0


# Train/validation/test split

In [ ]:
from sklearn.model_selection import train_test_split

ids = selected_ids

train_ids, temp_ids = train_test_split(
    ids,
    test_size=0.30,
    random_state=42,
    stratify=tracks.loc[ids, genre_column]
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=tracks.loc[temp_ids, genre_column]
)

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Train: 560
Validation: 120
Test: 120


In [ ]:
import json
import os

os.makedirs("/content/fma/splits", exist_ok=True)

with open("/content/fma/splits/train.json", "w") as f:
    json.dump(train_ids, f)

with open("/content/fma/splits/val.json", "w") as f:
    json.dump(val_ids, f)

with open("/content/fma/splits/test.json", "w") as f:
    json.dump(test_ids, f)

print("Splits saved.")

Splits saved.


# Text Preprocessing

In [ ]:
def get_text(track_id):
    row = tracks.loc[track_id]

    artist = str(row[("artist", "name")])
    album = str(row[("album", "title")])
    title = str(row[("track", "title")])

    return f"Artist: {artist}. Album: {album}. Track: {title}."
label_to_id = {
    genre: i for i, genre in enumerate(genres)
}

id_to_label = {
    i: genre for genre, i in label_to_id.items()
}

print(label_to_id)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)
from torch.utils.data import Dataset

class MusicTextDataset(Dataset):
    def __init__(self, track_ids):
        self.track_ids = track_ids

    def __len__(self):
        return len(self.track_ids)

    def __getitem__(self, idx):
        track_id = self.track_ids[idx]

        text = get_text(track_id)

        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=CONFIG["max_text_length"],
            return_tensors="pt"
        )

        label = label_to_id[
            tracks.loc[track_id, ("track", "genre_top")]
        ]

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": label,
            "track_id": track_id
        }


train_text_dataset = MusicTextDataset(train_ids)
val_text_dataset = MusicTextDataset(val_ids)
test_text_dataset = MusicTextDataset(test_ids)

print("Train:", len(train_text_dataset))
print("Validation:", len(val_text_dataset))
print("Test:", len(test_text_dataset))

{'Electronic': 0, 'Experimental': 1, 'Folk': 2, 'Hip-Hop': 3, 'Instrumental': 4, 'International': 5, 'Pop': 6, 'Rock': 7}
Train: 560
Validation: 120
Test: 120


# BERT

In [ ]:
from torch.utils.data import DataLoader

train_text_loader = DataLoader(
    train_text_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
)

val_text_loader = DataLoader(
    val_text_dataset,
    batch_size=CONFIG["batch_size"]
)

test_text_loader = DataLoader(
    test_text_dataset,
    batch_size=CONFIG["batch_size"]
)

In [ ]:
from transformers import AutoModel

class BERTClassifier(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            CONFIG["bert_model"]
        )

        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_classes
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        return_embedding=False
    ):
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        embedding = output.last_hidden_state[:, 0]

        logits = self.classifier(embedding)

        if return_embedding:
            return logits, embedding

        return logits


bert_model = BERTClassifier().to(device)

bert_optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=CONFIG["learning_rate"]
)

bert_criterion = nn.CrossEntropyLoss()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
for epoch in range(CONFIG["epochs"]):
    bert_model.train()

    total_loss = 0

    for batch in train_text_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        bert_optimizer.zero_grad()

        logits = bert_model(
            input_ids,
            attention_mask
        )

        loss = bert_criterion(
            logits,
            labels
        )

        loss.backward()
        bert_optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{CONFIG['epochs']} | "
        f"Loss: {total_loss / len(train_text_loader):.4f}"
    )

Epoch 1/3 | Loss: 1.6344
Epoch 2/3 | Loss: 0.7055
Epoch 3/3 | Loss: 0.3251


In [ ]:
bert_model.eval()

bert_true = []
bert_pred = []

with torch.no_grad():
    for batch in test_text_loader:
        logits = bert_model(
            batch["input_ids"],
            batch["attention_mask"]
        )

        predictions = logits.argmax(dim=1)

        bert_true.extend(
            batch["label"].numpy()
        )

        bert_pred.extend(
            predictions.numpy()
        )

print(
    "BERT Accuracy:",
    accuracy_score(bert_true, bert_pred)
)

print(
    "BERT Macro-F1:",
    f1_score(
        bert_true,
        bert_pred,
        average="macro"
    )
)

print(
    "BERT Micro-F1:",
    f1_score(
        bert_true,
        bert_pred,
        average="micro"
    )
)

BERT Accuracy: 0.7833333333333333
BERT Macro-F1: 0.7806319887594404
BERT Micro-F1: 0.7833333333333333
